In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/Unicorn_Companies.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()
print("\nDuplicate rows:", df.duplicated().sum())
print("\nMissing values:")
missing = df.isnull().sum()
print(pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(2)}))

In [ ]:
print(df["Financial Stage"].value_counts(dropna=False))
print()
print(df[["Valuation ($B)", "Total Raised"]].sample(8, random_state=1))
print()
print(df["Industry"].value_counts())

In [ ]:
ind = df["Industry"].value_counts()
print(ind.head(20))
print("\nDistinct Industry values:", len(ind))
print("Values appearing only once:", (ind == 1).sum())
print()
print(df[df["Industry"].isin(ind[ind == 1].index)].head(5).T)


In [ ]:
valid = ["Fintech", "Internet software & services", "E-commerce & direct-to-consumer",
         "Artificial intelligence", "Artificial Intelligence", "Health",
         "Supply chain, logistics, & delivery", "Other", "Cybersecurity",
         "Data management & analytics", "Mobile & telecommunications", "Hardware",
         "Auto & transportation", "Edtech", "Consumer & retail", "Travel", "Finttech"]
bad = df[~df["Industry"].isin(valid)]
print("Bad rows:", len(bad))
bad[["Company", "Valuation ($B)", "Country", "City", "Industry"]]

In [ ]:
print("Bad rows:", len(bad))
bad.head(3).T

In [ ]:
df = df.copy()
mask = ~df["Industry"].isin(valid)

df.loc[mask, "Select Inverstors"] = df.loc[mask, "Industry"]
df.loc[mask, "Industry"] = df.loc[mask, "City"]
df.loc[mask, "City"] = np.where(
    df.loc[mask, "Country"].isin(["Singapore", "Hong Kong"]),
    df.loc[mask, "Country"],
    np.nan,
)

# merge spelling variants
df["Industry"] = df["Industry"].replace({
    "Artificial Intelligence": "Artificial intelligence",
    "Finttech": "Fintech",
})

print("Distinct industries:", df["Industry"].nunique())
print(df["Industry"].value_counts())
df.loc[mask, ["Company", "Country", "City", "Industry", "Select Inverstors"]]

In [ ]:
# 1. Valuation: "$140" -> 140.0 (billions)
df["Valuation"] = df["Valuation ($B)"].str.replace("[$,]", "", regex=True).astype(float)

# 2. Total Raised: "$143M" / "$1.829B" -> millions
def to_millions(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace("$", "").replace(",", "").strip()
    if x.endswith("B"):
        return float(x[:-1]) * 1000
    if x.endswith("M"):
        return float(x[:-1])
    if x.endswith("K"):
        return float(x[:-1]) / 1000
    try:
        return float(x)
    except ValueError:
        return np.nan

df["Total Raised (M)"] = df["Total Raised"].apply(to_millions)

# 3. Dates and founding year
df["Date Joined"] = pd.to_datetime(df["Date Joined"], errors="coerce")
df["Join Year"] = df["Date Joined"].dt.year
df["Founded Year"] = df["Founded Year"].astype("Int64")
df["Years to Unicorn"] = df["Join Year"] - df["Founded Year"]

# 4. Target: exited (any Financial Stage value) vs still private
df["Exited"] = df["Financial Stage"].notna().astype(int)

# 5. Quick checks
print("Valuation nulls:", df["Valuation"].isna().sum())
print("Total Raised nulls:", df["Total Raised (M)"].isna().sum())
print("Date nulls:", df["Date Joined"].isna().sum())
print("Exited counts:\n", df["Exited"].value_counts())
print("Unparsed Total Raised:", df.loc[df["Total Raised (M)"].isna() & df["Total Raised"].notna(), "Total Raised"].unique())
print()
print(df[["Valuation", "Total Raised (M)", "Years to Unicorn"]].describe().round(2))


In [ ]:
odd = df[(df["Years to Unicorn"] < 0) | (df["Years to Unicorn"] > 40)]
print("Suspicious rows:", len(odd))
odd[["Company", "Founded Year", "Join Year", "Years to Unicorn", "Country"]].sort_values("Years to Unicorn")


In [10]:
# Impossible: founded after joining -> set founding info to missing
bad_year = df["Years to Unicorn"] < 0
df.loc[bad_year, ["Founded Year", "Years to Unicorn"]] = pd.NA

# Tidy column names for later notebooks
df = df.rename(columns={"Select Inverstors": "Select Investors"})

# Save
df.to_csv("../data/Unicorn_Companies_cleaned.csv", index=False)
print("Saved:", df.shape)
print("Years to Unicorn range:", df["Years to Unicorn"].min(), "to", df["Years to Unicorn"].max())


Saved: (1037, 18)
Years to Unicorn range: 0 to 98
